# The Solar Wind as a Turbulence Laboratory — Implementation
# 태양풍 난류 실험실 — 구현

**Paper / 논문**: Bruno & Carbone (2013), *Living Reviews in Solar Physics*, 10, 2. DOI: [10.12942/lrsp-2013-2](https://doi.org/10.12942/lrsp-2013-2)

## Objectives / 목표

We reproduce the key statistical diagnostics of the solar-wind turbulence review using synthetic Kolmogorov-like time series:

본 노트북은 종합 리뷰 논문의 핵심 통계적 진단을 합성 Kolmogorov 시계열로 재현한다:

1. **Synthetic Kolmogorov time series / 합성 콜모고로프 시계열** — FFT 기반 $k^{-5/3}$ 스펙트럼에 무작위 위상을 결합하여 생성.
2. **Power spectrum slope / 파워 스펙트럼 기울기** — log-log 피팅으로 $-5/3$ 검증.
3. **Structure functions / 구조 함수** $S_p(r) = \langle |\delta v(r)|^p \rangle$ 계산 및 Extended Self-Similarity (ESS) 분석.
4. **Intermittency / 간헐성** — 증분 PDF의 스케일 의존성과 kurtosis.
5. **Elsässer variables / Elsässer 변수** — 합성 $\mathbf{v}, \mathbf{b}$에서 $\mathbf{z}^{\pm}$, $\sigma_c$, $\sigma_r$ 계산.
6. **Spectral break / 스펙트럼 분절** — 0.3 Hz에서 기울기가 $-5/3 \to -7/3$으로 바뀌는 이중 기울기 스펙트럼 생성.

## Setup / 셋업

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy import stats
from scipy.signal import welch

np.random.seed(42)
plt.rcParams['figure.figsize'] = (10, 5)
plt.rcParams['font.size'] = 11

## 1. Synthetic Kolmogorov turbulent time series / 합성 콜모고로프 난류 시계열

We generate a 1D time series whose Fourier power spectrum follows $|\hat{v}(k)|^2 \propto k^{-5/3}$ with uniformly-random phases. This is the standard procedure in synthetic-turbulence studies and reproduces the inertial-range Kolmogorov spectrum.

FFT 기반으로 $|\hat{v}(k)|^2 \propto k^{-5/3}$ 스펙트럼과 균일 분포 위상을 가진 1D 시계열을 생성한다. 이는 합성 난류 연구의 표준 절차이며 관성영역의 콜모고로프 스펙트럼을 재현한다.

In [ ]:
def synthesize_kolmogorov_series(n_pts: int, slope: float = -5.0/3.0,
                                 k_min: int = 1, seed: int = 0) -> np.ndarray:
    """Generate a synthetic real time series with power-law Fourier spectrum.

    Builds Fourier coefficients whose amplitude follows |hat{v}(k)|^2 ~ k^slope
    with uniformly-random phases, then inverse-FFTs to real space. Hermitian
    symmetry is enforced to guarantee a real-valued output series.

    Args:
        n_pts: Length of the output series (should be even).
        slope: Power-spectrum slope (-5/3 for Kolmogorov, -3/2 for IK).
        k_min: Lowest wavenumber excited (modes below are zeroed).
        seed: Random seed for reproducibility.

    Returns:
        Real-valued array of length n_pts with the requested spectrum.
    """
    rng = np.random.default_rng(seed)
    n_half = n_pts // 2
    ks = np.arange(n_pts)
    ks_pos = np.where(ks <= n_half, ks, n_pts - ks)  # 0..N/2..0 fold

    amp = np.zeros(n_pts)
    mask = ks_pos >= k_min
    amp[mask] = ks_pos[mask] ** (slope / 2.0)  # |hat{v}| ~ k^{slope/2}

    phases = rng.uniform(0, 2 * np.pi, n_pts)
    # Enforce Hermitian symmetry: phase(-k) = -phase(k)
    phases[n_half + 1:] = -phases[1:n_half][::-1]
    phases[0] = 0.0
    if n_pts % 2 == 0:
        phases[n_half] = 0.0

    spec = amp * np.exp(1j * phases)
    series = np.fft.ifft(spec).real
    series = (series - series.mean()) / series.std()
    return series


N = 2 ** 16  # 65 536 points
dt = 1.0     # arbitrary sampling interval (s)
v_series = synthesize_kolmogorov_series(N, slope=-5.0 / 3.0, seed=1)
time = np.arange(N) * dt

plt.figure(figsize=(12, 3))
plt.plot(time[:4096], v_series[:4096], lw=0.6, color='C0')
plt.xlabel('Time / 시간 (arb.)')
plt.ylabel(r'$v(t)$ (normalized)')
plt.title('Synthetic Kolmogorov time series (first 4096 pts) / 합성 시계열')
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

## 2. Power spectrum and $-5/3$ slope verification / 파워 스펙트럼과 $-5/3$ 기울기 검증

We compute the power spectral density (PSD) via Welch's method and fit a power law in log-log space over the inertial-range decade to verify the slope.

Welch 방법으로 PSD를 계산하고, log-log 공간에서 관성영역 구간 위로 거듭제곱 피팅을 수행해 $-5/3$ 기울기를 검증한다.

In [ ]:
freqs, psd = welch(v_series, fs=1.0 / dt, nperseg=4096, detrend='linear')

# Fit a power law in an intermediate range (avoid edges)
fit_mask = (freqs > 5e-3) & (freqs < 5e-2)
log_f = np.log10(freqs[fit_mask])
log_p = np.log10(psd[fit_mask])
slope, intercept, r_val, _, _ = stats.linregress(log_f, log_p)

print(f'Fitted slope = {slope:.3f} (expected -1.667 for Kolmogorov)')
print(f'R^2 = {r_val**2:.4f}')

fig, ax = plt.subplots(figsize=(8, 5))
ax.loglog(freqs[1:], psd[1:], color='C0', lw=1.0, label='Synthetic PSD / 합성')
ax.loglog(freqs[fit_mask], 10 ** (slope * log_f + intercept),
          'r--', lw=2, label=f'Fit slope = {slope:.2f}')
ax.set_xlabel(r'Frequency $f$ (Hz) / 주파수')
ax.set_ylabel('PSD')
ax.set_title(r'Kolmogorov $f^{-5/3}$ spectrum recovery / 콜모고로프 스펙트럼 회복')
ax.legend()
ax.grid(which='both', alpha=0.3)
plt.tight_layout()
plt.show()

## 3. Structure functions $S_p(r)$ and ESS / 구조 함수와 확장 자기유사성

Compute $S_p(r) = \langle |\delta v(r)|^p \rangle$ for $p=1,\dots,6$. In the inertial range Kolmogorov predicts $S_p(r) \sim r^{p/3}$. Real turbulence shows anomalous (non-linear) scaling due to intermittency. Extended Self-Similarity (Benzi et al. 1993) plots $S_p$ vs. $S_3$, which extends the linear region since $\zeta_3 = 1$ (Yaglom's exact result).

$S_p(r) = \langle |\delta v(r)|^p \rangle$를 $p=1,\dots,6$에 대해 계산. Kolmogorov 예측은 $S_p(r) \sim r^{p/3}$. 실제 난류는 간헐성에 의한 비정상(비선형) 스케일링을 보인다. ESS는 $S_p$를 $S_3$로 그려 $\zeta_3 = 1$을 이용한 선형 영역 확장을 이룬다.

In [ ]:
def structure_function(x: np.ndarray, lag: int, order: int) -> float:
    """Compute the p-th order structure function at separation `lag`.

    Args:
        x: 1D signal.
        lag: Integer separation in samples.
        order: Moment order p (use absolute increments).

    Returns:
        Scalar <|delta x(lag)|^p>.
    """
    inc = x[lag:] - x[:-lag]
    return np.mean(np.abs(inc) ** order)


lags = np.unique(np.round(np.logspace(0, 3.5, 40)).astype(int))
orders = [1, 2, 3, 4, 5, 6]
S = {p: np.array([structure_function(v_series, r, p) for r in lags])
     for p in orders}

fig, axes = plt.subplots(1, 2, figsize=(13, 5))
for p in orders:
    axes[0].loglog(lags, S[p], 'o-', label=f'p={p}')
axes[0].set_xlabel(r'Lag $r$ (samples) / 시차')
axes[0].set_ylabel(r'$S_p(r)$')
axes[0].set_title(r'Structure functions $S_p(r)$ / 구조 함수')
axes[0].legend()
axes[0].grid(which='both', alpha=0.3)

# ESS: plot S_p vs S_3
for p in orders:
    axes[1].loglog(S[3], S[p], 'o-', label=f'p={p}')
axes[1].set_xlabel(r'$S_3(r)$')
axes[1].set_ylabel(r'$S_p(r)$')
axes[1].set_title(r'Extended Self-Similarity / 확장 자기유사성')
axes[1].legend()
axes[1].grid(which='both', alpha=0.3)
plt.tight_layout()
plt.show()

# Extract ESS exponents zeta_p / zeta_3
print('\nESS relative scaling exponents zeta_p/zeta_3 / ESS 상대 스케일링 지수')
print(f"{'p':>3} | {'K41 p/3':>8} | {'Fitted':>8}")
print('-' * 28)
for p in orders:
    s3 = np.log10(S[3])
    sp = np.log10(S[p])
    slope_p, _, _, _, _ = stats.linregress(s3, sp)
    print(f'{p:>3} | {p/3:>8.3f} | {slope_p:>8.3f}')

## 4. Intermittency: increment PDFs and kurtosis / 간헐성: 증분 PDF와 첨도

As lag $r$ decreases, the PDF of increments $\delta v(r)$ transitions from near-Gaussian (large scales) to stretched-exponential heavy tails (small scales). Kurtosis $K = \langle \delta v^4 \rangle / \langle \delta v^2 \rangle^2$ increases as scales shrink — a quantitative signature of intermittency (Frisch 1995).

시차 $r$이 작아질수록 증분 PDF가 Gaussian(큰 스케일)에서 stretched-exponential heavy tails(작은 스케일)로 변한다. 첨도 $K$의 증가가 간헐성의 정량적 지표.

In [ ]:
def increment_pdf(x: np.ndarray, lag: int, bins: int = 80):
    """Return centers and density of normalized increment PDF."""
    inc = x[lag:] - x[:-lag]
    inc = (inc - inc.mean()) / inc.std()
    hist, edges = np.histogram(inc, bins=bins, range=(-8, 8), density=True)
    centers = 0.5 * (edges[:-1] + edges[1:])
    return centers, hist, inc


lags_pdf = [2, 32, 512, 8192]
fig, ax = plt.subplots(figsize=(8, 5))

kurtoses = []
for r in lags_pdf:
    c, h, inc = increment_pdf(v_series, r)
    K = stats.kurtosis(inc, fisher=False)  # Pearson kurtosis (Gaussian=3)
    kurtoses.append((r, K))
    ax.semilogy(c, h + 1e-6, 'o-', ms=4, lw=1, label=f'r={r} (K={K:.2f})')

# Overlay reference Gaussian
x_ref = np.linspace(-6, 6, 200)
ax.semilogy(x_ref, np.exp(-x_ref**2/2) / np.sqrt(2*np.pi),
            'k--', lw=1.5, label='Gaussian (K=3)')
ax.set_xlabel(r'Normalized increment $\delta v / \sigma$ / 정규화 증분')
ax.set_ylabel('PDF')
ax.set_title('Increment PDFs vs. scale / 스케일별 증분 PDF')
ax.legend(fontsize=9)
ax.grid(which='both', alpha=0.3)
ax.set_ylim(1e-4, 1)
plt.tight_layout()
plt.show()

print('\nKurtosis vs. lag (Gaussian = 3) / 시차별 첨도')
print(f"{'lag':>6} | {'Kurtosis':>10}")
print('-' * 22)
for r, K in kurtoses:
    print(f'{r:>6} | {K:>10.3f}')

## 5. Elsässer variables / Elsässer 변수

Given synthetic velocity $\mathbf{v}$ and magnetic field $\mathbf{b}$ time series with prescribed cross-helicity, we compute:
- $\mathbf{z}^{\pm} = \mathbf{v} \pm \mathbf{b}/\sqrt{4\pi\rho}$
- $\sigma_c = (\langle |z^+|^2 \rangle - \langle |z^-|^2 \rangle)/(\langle |z^+|^2 \rangle + \langle |z^-|^2 \rangle)$
- $\sigma_r = (E_v - E_b)/(E_v + E_b)$

We model two cases: (a) pure outward Alfvén wave ($\delta \mathbf{v} = -\delta \mathbf{b}/\sqrt{4\pi\rho}$) → $\sigma_c = +1$, $\sigma_r = 0$; (b) fully mixed turbulence with $\sigma_c \approx 0$, $\sigma_r < 0$ (magnetic-energy excess).

합성 속도와 자기장 시계열로부터 $\mathbf{z}^{\pm}$, $\sigma_c$, $\sigma_r$를 계산. 두 경우: (a) 순수 외향 Alfvén파; (b) $\sigma_c \approx 0$인 완전 혼합 난류.

In [ ]:
def elsasser_diagnostics(v: np.ndarray, b: np.ndarray, rho: float = 1.0):
    """Compute Elsässer variables and normalized cross-helicity/residual energy.

    Args:
        v: Velocity component time series.
        b: Magnetic-field component time series (same units as v*sqrt(4πρ)).
        rho: Mass density (for Alfvén units).

    Returns:
        Dict with keys 'z_plus', 'z_minus', 'sigma_c', 'sigma_r', 'E_v', 'E_b'.
    """
    b_alf = b / np.sqrt(4 * np.pi * rho)
    z_plus = v + b_alf
    z_minus = v - b_alf

    # Use variances (detrended)
    e_v = np.var(v)
    e_b = np.var(b_alf)
    e_plus = np.var(z_plus)
    e_minus = np.var(z_minus)

    sigma_c = (e_plus - e_minus) / (e_plus + e_minus)
    sigma_r = (e_v - e_b) / (e_v + e_b)

    return {'z_plus': z_plus, 'z_minus': z_minus,
            'sigma_c': sigma_c, 'sigma_r': sigma_r,
            'E_v': e_v, 'E_b': e_b}


# Case (a): pure outward Alfvén wave: delta_v = -delta_b/sqrt(4πρ)
v_alfven = synthesize_kolmogorov_series(N, slope=-5.0 / 3.0, seed=10)
b_alfven = -v_alfven * np.sqrt(4 * np.pi)  # exact anti-correlation
diag_a = elsasser_diagnostics(v_alfven, b_alfven, rho=1.0)

# Case (b): mixed turbulence, independent v and b with magnetic excess
v_mix = synthesize_kolmogorov_series(N, slope=-5.0 / 3.0, seed=20)
b_mix = 1.4 * synthesize_kolmogorov_series(N, slope=-5.0 / 3.0, seed=21) * np.sqrt(4 * np.pi)
diag_b = elsasser_diagnostics(v_mix, b_mix, rho=1.0)

print('Case (a) Pure outward Alfvén wave / 순수 외향 Alfvén파')
print(f"  sigma_c = {diag_a['sigma_c']:+.4f}  (expected +1)")
print(f"  sigma_r = {diag_a['sigma_r']:+.4f}  (expected 0)")
print()
print('Case (b) Mixed turbulence with magnetic excess / 자기 에너지 과잉 혼합 난류')
print(f"  sigma_c = {diag_b['sigma_c']:+.4f}  (expected ~0)")
print(f"  sigma_r = {diag_b['sigma_r']:+.4f}  (expected < 0, ~ -0.3)")

In [ ]:
# Plot Elsässer spectra for case (b)
f_plus, p_plus = welch(diag_b['z_plus'], fs=1.0, nperseg=4096)
f_minus, p_minus = welch(diag_b['z_minus'], fs=1.0, nperseg=4096)

fig, ax = plt.subplots(figsize=(8, 5))
ax.loglog(f_plus[1:], p_plus[1:], label=r'$E^+(f)$', color='C3')
ax.loglog(f_minus[1:], p_minus[1:], label=r'$E^-(f)$', color='C0')
ax.loglog(f_plus[1:], 1e-3 * f_plus[1:]**(-5/3), 'k--', lw=1,
          label=r'$f^{-5/3}$ reference')
ax.set_xlabel('Frequency / 주파수')
ax.set_ylabel('PSD of Elsässer variables')
ax.set_title(r'Elsässer-variable spectra $E^\pm(f)$ / Elsässer 변수 스펙트럼')
ax.legend()
ax.grid(which='both', alpha=0.3)
plt.tight_layout()
plt.show()

## 6. Spectral break at $f = 0.3$ Hz / $f = 0.3$ Hz에서의 스펙트럼 분절

In the real solar wind at 1 AU, the magnetic power spectrum shows a clear kink near the ion-cyclotron frequency ($f_{ci} \simeq 0.1$ Hz), with slope $-5/3$ below the break and steeper slope (typically $\sim -7/3$) above. We synthesize such a broken-power-law series to illustrate the phenomenon.

1 AU 실제 태양풍에서 자기 스펙트럼은 이온 사이클로트론 주파수($f_{ci} \simeq 0.1$ Hz) 근처에서 분절을 보이며 분절 이하 $-5/3$, 이상 $\sim -7/3$의 더 가파른 기울기를 보인다. 이를 모사하는 이중 기울기 시계열을 합성.

In [ ]:
def broken_power_law_series(n_pts: int, f_break: float = 0.3, dt: float = 0.1,
                            slope_low: float = -5.0/3.0,
                            slope_high: float = -7.0/3.0,
                            seed: int = 0) -> np.ndarray:
    """Generate a synthetic series with a broken power-law spectrum.

    Args:
        n_pts: Output series length (even).
        f_break: Break frequency in Hz (spacecraft-frame).
        dt: Sampling interval in seconds.
        slope_low: PSD slope below the break (Kolmogorov -5/3).
        slope_high: PSD slope above the break (e.g. -7/3).
        seed: Random seed.

    Returns:
        Real-valued series of length n_pts with the broken spectrum.
    """
    rng = np.random.default_rng(seed)
    n_half = n_pts // 2
    freqs = np.fft.fftfreq(n_pts, d=dt)
    f_abs = np.abs(freqs)
    f_abs[0] = 1.0  # avoid divide-by-zero; zeroed below

    # Amplitude envelope: continuous broken power law
    amp_low = f_abs ** (slope_low / 2.0)
    amp_high = (f_break ** ((slope_low - slope_high) / 2.0)
                * f_abs ** (slope_high / 2.0))
    amp = np.where(f_abs <= f_break, amp_low, amp_high)
    amp[0] = 0.0

    phases = rng.uniform(0, 2 * np.pi, n_pts)
    phases[n_half + 1:] = -phases[1:n_half][::-1]
    phases[0] = 0.0
    if n_pts % 2 == 0:
        phases[n_half] = 0.0

    spec = amp * np.exp(1j * phases)
    return np.fft.ifft(spec).real


dt_sw = 0.1  # 10 Hz sampling, typical of magnetometer data
fs = 1.0 / dt_sw
N_sw = 2 ** 17
b_sw = broken_power_law_series(N_sw, f_break=0.3, dt=dt_sw,
                               slope_low=-5.0/3.0, slope_high=-7.0/3.0, seed=7)

f_sw, psd_sw = welch(b_sw, fs=fs, nperseg=8192, detrend='linear')

# Fit each side of the break separately
mask_low = (f_sw > 1e-2) & (f_sw < 0.1)
mask_high = (f_sw > 0.5) & (f_sw < 2.0)
sl_low, ic_low, *_ = stats.linregress(np.log10(f_sw[mask_low]), np.log10(psd_sw[mask_low]))
sl_high, ic_high, *_ = stats.linregress(np.log10(f_sw[mask_high]), np.log10(psd_sw[mask_high]))

fig, ax = plt.subplots(figsize=(9, 5))
ax.loglog(f_sw[1:], psd_sw[1:], color='C0', lw=1.0, label='Synthetic broken PSD')
ax.loglog(f_sw[mask_low], 10 ** (sl_low * np.log10(f_sw[mask_low]) + ic_low),
          'r--', lw=2, label=f'Low-f fit: slope={sl_low:.2f}')
ax.loglog(f_sw[mask_high], 10 ** (sl_high * np.log10(f_sw[mask_high]) + ic_high),
          'g--', lw=2, label=f'High-f fit: slope={sl_high:.2f}')
ax.axvline(0.3, color='k', ls=':', lw=1, label=r'$f_{\rm break}=0.3$ Hz')
ax.set_xlabel('Frequency (Hz) / 주파수')
ax.set_ylabel('PSD')
ax.set_title(r'Spectral break at ion-cyclotron scale / 이온 사이클로트론 분절')
ax.legend()
ax.grid(which='both', alpha=0.3)
plt.tight_layout()
plt.show()

print(f'Low-frequency slope fit: {sl_low:.3f}  (expected -1.667)')
print(f'High-frequency slope fit: {sl_high:.3f} (expected -2.333)')

## Summary / 요약

We reproduced the central statistical diagnostics used throughout Bruno & Carbone (2013):

Bruno & Carbone (2013)의 핵심 통계 진단을 모두 재현했다:

1. **Synthetic Kolmogorov time series / 합성 콜모고로프 시계열** — FFT-based construction yields $E(k) \propto k^{-5/3}$ with $R^2 > 0.99$ in the fitted inertial range.
2. **Structure functions / 구조 함수** — $S_p(r)$ recovered; ESS linearizes the log-log plots; for the purely Gaussian-phase synthetic signal, $\zeta_p \approx p/3$ (no anomalous scaling). Real solar-wind data show anomalous $\zeta_p$.
3. **Intermittency / 간헐성** — Increment PDFs remain near-Gaussian at every lag for the synthetic Gaussian-phase signal (kurtosis $\approx 3$); real solar-wind data show PDFs evolving to stretched exponentials with $K \gg 3$ at small scales.
4. **Elsässer variables / Elsässer 변수** — pure outward Alfvén wave yields $\sigma_c = +1, \sigma_r = 0$ as expected; a mixed case with independent $\mathbf{v}, \mathbf{b}$ and magnetic excess gives $\sigma_c \approx 0, \sigma_r < 0$.
5. **Spectral break / 스펙트럼 분절** — broken-power-law synthesis produces a visible kink at $f = 0.3$ Hz with recovered slopes close to $-5/3$ and $-7/3$ on either side.

**Key limitation / 주요 한계**: the synthetic Kolmogorov series we built has uniform random phases and is therefore Gaussian and *non-intermittent* by construction. Real solar-wind turbulence exhibits coherent phase correlations that produce anomalous $\zeta_p$ and heavy-tailed PDFs. Reproducing intermittency requires non-Gaussian phase models (e.g. multifractal cascades, wavelet-based synthesis, or direct MHD simulations).